In [1]:
# Docker container : https://hub.docker.com/r/continumio/anaconda3
# rdkit installation: https://www.rdkit.org/docs/Install.html

In [2]:
import mols2grid
import pandas as pd
import streamlit as st
import streamlit.components.v1 as components
from rdkit import Chem

from rdkit.Chem import rdMolDescriptors # opt

from rdkit.Chem.Descriptors import ExactMolWt, MolLogP, NumHDonors, NumHAcceptors

In [3]:
st.title(" Filter FDA Approved Drugs by Lipinski's Rule-of-five with Streamlit")

2025-12-30 20:23:47.364 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:48.252 
  command:

    streamlit run C:\Users\nachiket\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-12-30 20:23:48.253 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:48.254 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [4]:
# !pip install mols2grid

In [5]:
#import mols2grid

# opt added for testing not the actual code

# Simple test with Caffeine
#smiles_list = ["CN1C=NC2=C1C(=O)N(C(=O)N2C)C"]
#df = pd.DataFrame({"SMILES": smiles_list})

#mols2grid.display(df, smiles_col="SMILES")

In [6]:
st.markdown("""
- App modified by [chanin nantasenamat](http://medicum.dataprofessor.org) (aka[data])
- Original app by [justin chavez] (https://blog.reverielabs.com/building-web-application)
""")

2025-12-30 20:23:48.602 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:48.605 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:48.608 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [7]:
@st.cache(allow_output_mutation = True)
def download_dataset():
    """Loads once then cached for subsequent runs"""
    df = pd.read_csv(
    "https://www.cureffi.org/wp-content/uploads/2013/10/drugs.txt", sep = "\t"
    ).dropna()
    return df

2025-12-30 20:23:48.771 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:48.774 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:48.777 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:48.781 
`st.cache` is deprecated and will be removed soon. Please use one of Streamlit's new
caching commands, `st.cache_data` or `st.cache_resource`. More information
[in our docs](https://docs.streamlit.io/develop/concepts/architecture/caching).

**Note**: The behavior of `st.cache` was updated in Streamlit 1.36 to the new caching
logic used by `st.cache_data` and `st.cache_resource`. This might lead to some problems
or unexpected behavior in certain edge cases.



In [8]:
# calculate desriptors
def calc_mw(smiles_string):
    """ Given a smiles string (ex. c1cccccc1), calculate and return the molecular weight"""
    mol = Chem.MolFromSmiles(smiles_string)
    return ExactMolWt(mol)

In [9]:
def calc_logp(smiles_string):
    """ Given a smiles string (ex. c1cccccc1), calculate and return the Logp"""
    mol = Chem.MolFromSmiles(smiles_string)
    return MolLogP(mol)

In [10]:
def calc_NumHDonors(smiles_string):
    """ Given a smiles string (ex. c1cccccc1), calculate and return the NumHDonors"""
    mol = Chem.MolFromSmiles(smiles_string)
    return NumHDonors(mol)

In [11]:
def calc_NumHAcceptors(smiles_string):
    """ Given a smiles string (ex. c1cccccc1), calculate and return the NumHAcceptors"""
    mol = Chem.MolFromSmiles(smiles_string)
    return NumHAcceptors(mol)

In [12]:
# copy the dataset so any changes are not applied to the original cached version
df = download_dataset().copy()
df["MW"] = df.apply(lambda x: calc_mw(x["smiles"]), axis = 1)
df["LogP"] = df.apply(lambda x: calc_logp(x["smiles"]), axis = 1)
df["NumHDonors"] = df.apply(lambda x: calc_NumHDonors(x["smiles"]), axis = 1)
df["NumHAcceptors"] = df.apply(lambda x:  calc_NumHAcceptors(x["smiles"]), axis = 1)

2025-12-30 20:23:49.448 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:49.450 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:49.454 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:49.457 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:49.962 Thread 'Thread-3': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:49.970 Thread 'Thread-3': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:49.973 Thread 'Thread-3': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:50.225 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode

In [13]:
# sidebar panel
st.sidebar.header('Set parameters')
st.sidebar.write('*Note: Display compounds having values less than the following the')
weight_cutoff = st.sidebar.slider(
    label = " Molecular weight",
    min_value = 0,
    max_value = 10000,
    value = 500,
    step = 10,
)
logp_cutoff = st.sidebar.slider(
    label = " LogP",
    min_value = 10,
    max_value = 10,
    value = 5,
    step = 1,
)
NumHDonors_cutoff = st.sidebar.slider(
    label = " NumHDonors",
    min_value = 0,
    max_value = 15,
    value = 5,
    step = 1,
)
NumHAcceptors_cutoff = st.sidebar.slider(
    label = " NumHAcceptors",
    min_value = 0,
    max_value = 20,
    value = 10,
    step = 1,
)

2025-12-30 20:23:55.603 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:55.605 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:55.607 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:55.611 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:55.613 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:55.615 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:55.617 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:55.619 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [14]:
df_result = df[df["MW"] < weight_cutoff]
df_result2 = df_result[df_result["LogP"] < logp_cutoff]
df_result3 = df_result2[df_result2["NumHDonors"] < NumHDonors_cutoff]
df_result4 = df_result3[df_result3["NumHAcceptors"] < NumHAcceptors_cutoff]

In [15]:
st.write(df_result4.shape)
st.write(df_result4)

2025-12-30 20:23:55.891 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:55.894 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:55.897 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:55.901 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:55.904 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:55.914 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:55.933 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:23:55.934 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [24]:
# Add this to your sidebar for debugging # opt
print(st.sidebar.subheader("Diagnostic: Column Names"))
print(st.sidebar.write(list(df_result4.columns)))

2025-12-30 20:30:03.955 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:30:03.957 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:30:03.959 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:30:03.964 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:30:03.966 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:30:03.968 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator(_root_container=1, _parent=DeltaGenerator())
None


In [28]:
import streamlit.components.v1 as components # opt

# Ensure columns are exactly what we expect
# This renames them if they are lowercase or have different names
df_to_display = df_result4.rename(columns={
    "smiles": "SMILES",
    "SMILES": "SMILES",
    "generic_name": "Name",
    "name": "Name"
})

try:
    # We use 'SMILES' as the anchor for the grid
 # Create the grid object
 grid = mols2grid.display(
    df_result4,
    subset=["Name", "MW"],
    mapping={"SMILES": "SMILES", "Name": "Name"}
)

# Render it using the object's own internal HTML string

 components.html(raw_html, width=900, height=1100, scrolling=False)

except KeyError as e:
    st.error(f"Mapping Error: The column {e} was not found in the dataframe.")
    st.write("Available columns are:", list(df_result4.columns))

2025-12-30 20:32:19.157 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:32:19.160 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:32:19.164 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:32:19.168 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:32:19.171 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:32:19.173 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:32:19.176 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 20:32:19.178 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [29]:
raw_html = mols2grid.display(# main code why not working
    df_result4, 
    subset = ["Name", "img", "MW"], 
    mapping = {"smiles": "SMILES", "generic_name": "Name"})._repr_html() 
components.html(raw_html, width = 900, height = 1100, scrolling = False)

KeyError: "['SMILES', 'Name'] not in index"